# Demo 03 : Routing

Run the cells in order and watch. There is nothing to write.

## a · conditional-edges

We route a build from its bug list. No bugs : ship. Any bug : back to the Coder.

In [1]:
from typing import Literal, TypedDict
from langgraph.graph import END, START, StateGraph

class Board(TypedDict):
    bugs: list[str]
    tests: str
    outcome: str

def route(board: Board) -> Literal["ship", "coder"]:
    return "coder" if board["bugs"] else "ship"

builder = StateGraph(Board)
builder.add_node("tester", lambda board: {})
builder.add_node("ship", lambda board: {"outcome": "shipped"})
builder.add_node("coder", lambda board: {"outcome": "back to the coder"})
builder.add_edge(START, "tester")
builder.add_conditional_edges("tester", route)
builder.add_edge("ship", END)
builder.add_edge("coder", END)
graph = builder.compile()
for bugs in ([], ["self-hit"]):
    print(bugs, "->", graph.invoke({"bugs": bugs, "tests": "", "outcome": ""})["outcome"])

[] -> shipped
['self-hit'] -> back to the coder


## b · path-map

We map router labels to node names. The label says what happened, and the node name says who works next.

In [2]:
mapped = StateGraph(Board)
mapped.add_node("tester", lambda board: {})
mapped.add_node("ship", lambda board: {"outcome": "shipped"})
mapped.add_node("coder", lambda board: {"outcome": "back to the coder"})
mapped.add_edge(START, "tester")
mapped.add_conditional_edges("tester", lambda board: board["tests"], {"pass": "ship", "fail": "coder"})
mapped.add_edge("ship", END)
mapped.add_edge("coder", END)
mapped_graph = mapped.compile()
print("mapped:", mapped_graph.invoke({"bugs": [], "tests": "fail", "outcome": ""}))
try:
    mapped_graph.invoke({"bugs": [], "tests": "flaky", "outcome": ""})
except KeyError as exc:
    print("unmapped label:", type(exc).__name__, exc)

mapped: {'bugs': [], 'tests': 'fail', 'outcome': 'back to the coder'}
unmapped label: KeyError 'flaky'


## c · recursion-limit

The Coder sends every build straight back to itself, with no router. We cap the run at two supersteps and display
LangGraph's recursion error. Then we run the same loop with no `recursion_limit` and count the steps before the
default limit stops it.

In [3]:
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph

class Retry(TypedDict):
    attempts: int

retry = StateGraph(Retry)
retry.add_node("coder", lambda board: {"attempts": board["attempts"] + 1})
retry.add_edge(START, "coder")
retry.add_edge("coder", "coder")
try:
    retry.compile().invoke({"attempts": 0}, {"recursion_limit": 2})
except GraphRecursionError as exc:
    print(type(exc).__name__)

GraphRecursionError


In [4]:
steps = 0
try:
    for _ in retry.compile().stream({"attempts": 0}):
        steps += 1
except GraphRecursionError as exc:
    print("steps without a limit:", f"{steps:,}")
    print(str(exc).split(".")[0])

steps without a limit: 10,007
Recursion limit of 10007 reached without hitting a stop condition


## Things to try

- Add a third label, `flaky`, to the `path_map`
- Give the retry graph a conditional exit